# 1) Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("✓ Schema Gold preparado")

✓ Schema Gold preparado


# 2) gold.dim_movies
Cria a dimensão de filmes com chave substituta (surrogate key).

In [0]:
janela = Window.orderBy("id_filme")

df_dim_movies = (
    spark.table("workspace.silver.tb_info_filmes")
    .withColumn(
        "sk_movie_id",
        F.row_number().over(janela).cast("bigint")
    )
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

assert (
    df_dim_movies.count()
    == df_dim_movies.select("sk_movie_id").distinct().count()
), "dim_movies possui chaves substitutas duplicadas"

(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_movies")
)

print("✓ gold.dim_movies concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold.dim_movies concluída


# 3) gold.dim_genres
Catálogo único de gêneros com surrogate key.

In [0]:
df_generos = spark.table("workspace.silver.tb_generos")

janela_generos = Window.orderBy("nome_genero")

df_dim_genres = (
    df_generos
    .select("nome_genero")
    .distinct()
    .withColumn(
        "sk_genre_id",
        F.row_number().over(janela_generos).cast("bigint")
    )
    .select(
        "sk_genre_id",
        "nome_genero"
    )
)

(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_genres")
)

print("✓ gold.dim_genres concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold.dim_genres concluída


# 4) gold.bridge_movie_genre
Relaciona filmes e gêneros sem duplicar o grão da tabela fato.

In [0]:
df_bridge_movie_genre = (
    spark.table("workspace.silver.tb_generos")

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )

    .join(
        df_dim_genres,
        "nome_genero",
        "inner"
    )

    .select(
        "sk_movie_id",
        "sk_genre_id"
    )

    .dropDuplicates()
)

(
    df_bridge_movie_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_genre")
)

print("✓ gold.bridge_movie_genre concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold.bridge_movie_genre concluída


# 5) gold.dim_people
Catálogo deduplicado de pessoas físicas com surrogate key.


In [0]:
df_pessoas = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
)

janela_pessoas = Window.orderBy("nome_pessoa", "tipo_pessoa")

df_dim_people = (
    df_pessoas
    .withColumn(
        "sk_person_id",
        F.row_number().over(janela_pessoas).cast("bigint")
    )
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa"
    )
)

(
    df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_people")
)

print("✓ gold.dim_people concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold.dim_people concluída


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


# 6) gold.bridge_movie_person
Relaciona filmes às pessoas envolvidas sem duplicar a tabela fato.

In [0]:
df_bridge_movie_person = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )

    .join(
        spark.table("workspace.gold.dim_movies")
        .select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )

    .join(
        spark.table("workspace.gold.dim_people"),
        ["nome_pessoa", "tipo_pessoa"],
        "inner"
    )

    .select(
        "sk_movie_id",
        "sk_person_id"
    )
    .dropDuplicates()
)

(
    df_bridge_movie_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_person")
)

print("✓ gold.bridge_movie_person concluída")

✓ gold.bridge_movie_person concluída


# 7) gold.dim_companies
Catálogo único de produtoras/estúdios com surrogate key.

In [0]:
df_produtoras = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("nome_entidade").alias("nome_produtora")
    )
    .distinct()
)

janela_produtoras = Window.orderBy("nome_produtora")

df_dim_companies = (
    df_produtoras
    .withColumn(
        "sk_company_id",
        F.row_number().over(janela_produtoras).cast("bigint")
    )
    .select(
        "sk_company_id",
        "nome_produtora"
    )
)

(
    df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_companies")
)

print("✓ gold.dim_companies concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold.dim_companies concluída


# 8) gold.bridge_movie_company
Relaciona filmes às produtoras sem duplicar o grão da fato.

In [0]:
df_bridge_movie_company = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_produtora")
    )

    .join(
        spark.table("workspace.gold.dim_movies")
        .select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )

    .join(
        spark.table("workspace.gold.dim_companies"),
        "nome_produtora",
        "inner"
    )

    .select(
        "sk_movie_id",
        "sk_company_id"
    )
    .dropDuplicates()
)

(
    df_bridge_movie_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.bridge_movie_company")
)

print("✓ gold.bridge_movie_company concluída")

✓ gold.bridge_movie_company concluída


# 9) gold.dim_reviews
Resume as avaliações dos usuários por filme.

In [0]:
df_reviews = (
    spark.table("workspace.silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2)
         .cast("double")
         .alias("nota_media_usuarios")
    )

    .join(
        spark.table("workspace.gold.dim_movies")
        .select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )
)

janela_reviews = Window.orderBy("sk_movie_id")

df_dim_reviews = (
    df_reviews
    .withColumn(
        "sk_review_id",
        F.row_number().over(janela_reviews).cast("bigint")
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_reviews")
)

print("✓ gold.dim_reviews concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold.dim_reviews concluída


# 10) gold.fact_movies_performance
Consolida métricas financeiras e de engajamento.

O grão da tabela é um único registro por filme lançado.

In [0]:
df_fact_movies = (
    spark.table("workspace.gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")

    .join(
        spark.table("workspace.silver.tb_financeiro_filmes"),
        "id_filme",
        "left"
    )

    .join(
        spark.table("workspace.silver.tb_metricas_engajamento"),
        "id_filme",
        "left"
    )

    .select(
        "sk_movie_id",

        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),

        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),

        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
)

assert (
    df_fact_movies.count()
    == df_fact_movies.select("sk_movie_id").distinct().count()
), "fact_movies_performance possui filmes duplicados"

(
    df_fact_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_movies_performance")
)

print("✓ gold.fact_movies_performance concluída")

✓ gold.fact_movies_performance concluída


# 11) gold.gold_genai_movies_context
Consolida informações do filme em um texto próprio para vetorização/RAG.

coalesce() evita que campos nulos façam o documento inteiro virar NULL.

In [0]:
df_pessoas_filme = (
    spark.table("workspace.gold.bridge_movie_person")
    .join(
        spark.table("workspace.gold.dim_people"),
        "sk_person_id",
        "inner"
    )
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set(
                    F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa"))
                )
            )
        ).alias("atores"),

        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set(
                    F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa"))
                )
            )
        ).alias("diretores")
    )
)

df_genai = (
    spark.table("workspace.gold.dim_movies")
    .join(
        spark.table("workspace.gold.fact_movies_performance"),
        "sk_movie_id",
        "left"
    )
    .join(
        df_pessoas_filme,
        "sk_movie_id",
        "left"
    )
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),

        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("titulo"), F.lit("Título não informado")),
            F.lit(", lançado no ano de "),
            F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")),
            F.lit(", faturou "),
            F.coalesce(
                F.concat(F.lit("US$ "), F.col("receita_usd").cast("string")),
                F.lit("valor não informado")
            ),
            F.lit(" e teve um custo de "),
            F.coalesce(
                F.concat(F.lit("US$ "), F.col("orcamento_usd").cast("string")),
                F.lit("valor não informado")
            ),
            F.lit(". Estrelado por "),
            F.when(
                F.col("atores").isNull() | (F.col("atores") == ""),
                F.lit("elenco não informado")
            ).otherwise(F.col("atores")),
            F.lit(" e dirigido por "),
            F.when(
                F.col("diretores").isNull() | (F.col("diretores") == ""),
                F.lit("diretor não informado")
            ).otherwise(F.col("diretores")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.col("sinopse"), F.lit("sinopse não informada")),
            F.lit(".")
        ).alias("llm_context_document")
    )
)

(
    df_genai.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_genai_movies_context")
)

print("✓ gold.gold_genai_movies_context concluída")

✓ gold.gold_genai_movies_context concluída


# 12) Analytics - Perguntas de negócio

In [0]:
fact = spark.table("workspace.gold.fact_movies_performance")
movies = spark.table("workspace.gold.dim_movies")

# Data limite para os recortes de 2 e 5 anos:
# lançamento realizado mais recente, ignorando datas futuras.
data_limite = (
    movies
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.current_date())
    )
    .agg(F.max("data_lancamento"))
    .first()[0]
)

print(f"Data limite utilizada: {data_limite}")


# 1. Receita total em R$
display(
    fact.agg(
        F.sum("receita_brl").alias("receita_total_brl")
    )
)


# 2. Top 5 filmes com maior popularidade
display(
    fact
    .join(movies.select("sk_movie_id", "titulo"), "sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(F.desc("popularidade"))
    .limit(5)
)


# 3. Quantidade de filmes por gênero
display(
    spark.table("workspace.gold.bridge_movie_genre")
    .join(
        spark.table("workspace.gold.dim_genres"),
        "sk_genre_id"
    )
    .groupBy("nome_genero")
    .agg(
        F.countDistinct("sk_movie_id").alias("qtd_filmes")
    )
    .orderBy(F.desc("qtd_filmes"))
)


# 4. Top 10 filmes por receita, com RANK()
janela_receita = Window.orderBy(F.desc("receita_usd"))

display(
    fact
    .join(movies.select("sk_movie_id", "titulo"), "sk_movie_id")
    .filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao", F.rank().over(janela_receita))
    .select(
        "posicao",
        "titulo",
        "receita_usd",
        "receita_brl"
    )
    .orderBy("posicao")
    .limit(10)
)


# 5. Ator com mais participações nos últimos 2 anos
atores_2_anos = (
    spark.table("workspace.gold.bridge_movie_person")
    .join(
        spark.table("workspace.gold.dim_people")
        .filter(F.col("tipo_pessoa") == "Ator"),
        "sk_person_id"
    )
    .join(
        movies.select("sk_movie_id", "data_lancamento", "status_filme"),
        "sk_movie_id"
    )
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.lit(data_limite)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -24))
    )
    .groupBy("nome_pessoa")
    .agg(
        F.countDistinct("sk_movie_id").alias("qtd_participacoes")
    )
)

janela_atores = Window.orderBy(F.desc("qtd_participacoes"))

display(
    atores_2_anos
    .withColumn("posicao", F.rank().over(janela_atores))
    .filter(F.col("posicao") == 1)
    .select("nome_pessoa", "qtd_participacoes")
)


# 6. Produtora com maior lucro nos últimos 5 anos
produtoras_5_anos = (
    spark.table("workspace.gold.bridge_movie_company")
    .join(
        spark.table("workspace.gold.dim_companies"),
        "sk_company_id"
    )
    .join(
        movies.select("sk_movie_id", "data_lancamento", "status_filme"),
        "sk_movie_id"
    )
    .join(
        fact.select("sk_movie_id", "lucro_usd"),
        "sk_movie_id"
    )
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.lit(data_limite)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -60)) &
        F.col("lucro_usd").isNotNull()
    )
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd")
    )
)

janela_produtoras = Window.orderBy(F.desc("lucro_total_usd"))

display(
    produtoras_5_anos
    .withColumn("posicao", F.rank().over(janela_produtoras))
    .filter(F.col("posicao") == 1)
    .select("nome_produtora", "lucro_total_usd")
)

Data limite utilizada: 2026-02-19


receita_total_brl
837771066819.07


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


nome_genero,qtd_filmes
Drama,32251
Documentary,18983
Comedy,18588
Thriller,10266
Horror,9716
Romance,7628
Action,6035
Crime,4733
Animation,4459
TV Movie,4071


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14439320000.00
2,Avatar: The Way of Water,2320250281.00,11965298674.09
3,AVENGERS: INFINITY WAR,2052415039.00,10584099114.62
4,spider-man: no way home,1921847111.00,9910773366.72
5,The Lion King,1663075401.00,8576313535.42
6,Top Gun: Maverick,1488732821.00,7677246284.61
7,Barbie,1428545028.00,7366863854.89
8,The Super Mario Bros. Movie,1355725263.00,6991339608.76
9,Black Panther,1349926083.00,6961433817.42
10,Star Wars: The Last Jedi,1332698830.00,6872594596.43


nome_pessoa,qtd_participacoes
Kevin Hart,64


nome_produtora,lucro_total_usd
Universal Pictures,5772329679.00


# 13) Validação final da camada Gold

In [0]:
tabelas_gold = [
    "dim_movies",
    "dim_genres",
    "bridge_movie_genre",
    "dim_people",
    "bridge_movie_person",
    "dim_companies",
    "bridge_movie_company",
    "dim_reviews",
    "fact_movies_performance",
    "gold_genai_movies_context"
]

# Todas as tabelas devem existir e possuir registros.
for tabela in tabelas_gold:
    nome = f"workspace.gold.{tabela}"

    assert spark.catalog.tableExists(nome), f"{tabela} não foi criada"

    qtd = spark.table(nome).count()
    assert qtd > 0, f"{tabela} está vazia"

    print(f"✓ {tabela}: {qtd:,} registros")


# Chaves primárias / surrogate keys devem ser únicas.
validacoes_unicidade = [
    ("dim_movies", "sk_movie_id"),
    ("dim_genres", "sk_genre_id"),
    ("dim_people", "sk_person_id"),
    ("dim_companies", "sk_company_id"),
    ("dim_reviews", "sk_review_id"),
    ("fact_movies_performance", "sk_movie_id")
]

for tabela, chave in validacoes_unicidade:
    df = spark.table(f"workspace.gold.{tabela}")

    assert (
        df.count() == df.select(chave).distinct().count()
    ), f"{tabela} possui {chave} duplicada"


# Cada filme deve ter no máximo um resumo de reviews.
df_reviews = spark.table("workspace.gold.dim_reviews")

assert (
    df_reviews.count()
    == df_reviews.select("sk_movie_id").distinct().count()
), "dim_reviews possui mais de um registro por filme"


# Bridges não podem repetir a mesma relação.
for tabela, chaves in [
    ("bridge_movie_genre", ["sk_movie_id", "sk_genre_id"]),
    ("bridge_movie_person", ["sk_movie_id", "sk_person_id"]),
    ("bridge_movie_company", ["sk_movie_id", "sk_company_id"])
]:
    df = spark.table(f"workspace.gold.{tabela}")

    assert (
        df.count() == df.select(*chaves).distinct().count()
    ), f"{tabela} possui relações duplicadas"


# O documento destinado ao RAG nunca pode ser nulo.
df_genai = spark.table("workspace.gold.gold_genai_movies_context")

assert (
    df_genai.filter(F.col("llm_context_document").isNull()).count() == 0
), "Existem documentos GenAI nulos"


print("\n✓ Camada Gold validada com sucesso")

✓ dim_movies: 97,879 registros
✓ dim_genres: 19 registros
✓ bridge_movie_genre: 140,243 registros
✓ dim_people: 420,597 registros
✓ bridge_movie_person: 767,396 registros
✓ dim_companies: 45,266 registros
✓ bridge_movie_company: 118,326 registros
✓ dim_reviews: 27,303 registros
✓ fact_movies_performance: 96,463 registros
✓ gold_genai_movies_context: 97,879 registros

✓ Camada Gold validada com sucesso
